In [1]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma
from langchain_openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
import os
from langchain_core.messages import HumanMessage

D:\code floders\PYTHON-5\LANGCHAIN_MODELS\new_environment\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-607594f802f568b7f0d9db662bc044e72ffd5"

In [3]:
model = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="openrouter/free",
    temperature=0.7,
)

In [4]:
import operator
from typing import TypedDict , List , Annotated

from pydantic import BaseModel , Field


In [7]:
class Task(BaseModel):
    id:int
    title:str
    brief : str = Field(... , description = "what to cover")
    

In [8]:
class Plan(BaseModel):
    blog_title : str
    tasks : List[Task]

In [9]:
class State(TypedDict):
    topic : str
    plan : Plan
    # reducer : results from workers get concatenated automatically

    sections : Annotated[List[str] , operator.add]
    final:str

In [11]:
def orchestrator(state: dict) -> dict:
    plan_response = model.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content="Create a blog plan with 5-7 sections on the following topic."
            ),
            HumanMessage(
                content=f"Topic: {state['topic']}"
            ),
        ]
    )

    return {"plan": plan_response}

In [ ]:
def fanout(state:State):

    return [Send("worker",{"task":task , "topic":state["topics"] , "plan":state["plan"]})
               for task in state["plan"].tasks]
    

In [ ]:
def worker(payload: dict) -> dict:

    # payload contains what we want

    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    blog_title = plan.blog_title

    section_md = model.invoke(
        [
            SystemMessage(content= "write one clean markdown section"),
            HumanMessage(
                content = (
                    f"Blog: {blog_title}\n"
                    f"Topic: {topic}\n\n"
                    f"Section: {task.title}\n"
                    f"Brief:{task.brief}\n"
                    "Return only the section in content in markdown."
                    
                    
                )
            )
        ]
    ).content.strip()